# Paper 2 — Exact Workflow Mirror (real pipeline, full model check, controlled max-power run)

**This replaces the earlier version.** That one paraphrased the real workflow
and got a real step wrong (it skipped `_load_waste_heat()`'s Stadtbach-specific
path and would have silently used the wrong COP source-temperature series for
Stadtbach). This version does not paraphrase anything: every code cell below
is copied, step-for-step, from `scripts/paper_2/scenario_runner.py::run_single_scenario()`
and `calion/run/solver.py::_solve_scenario()` -- same functions, same order,
same conditionals, same variable names where practical, cross-referenced by
line number against the real file so you can diff it yourself at any time.

**Structure**:
- **Part A (Cells 1-8)**: the exact config-preparation pipeline
  (load -> overrides -> TES/HP siting -> DSM -> endogenous siting -> data
  table -> heating curve -> COP (with the real waste-heat-first-then-fallback
  logic) -> spatial offsets), each cell labeled with its source line range.
- **Part B (Cells 9-11)**: build the model and **exhaustively check it**
  before running anything -- every variable group, every constraint group,
  every mutable parameter, the full objective decomposition, plus explicit
  PASS/FAIL assertions. Nothing here costs real compute (build only, no solve).
- **Part C (Cell 12)**: the **real solve** -- calls `run_workflow()`, the
  exact function the campaign uses (not a hand-assembled solver call). This
  necessarily rebuilds the model internally (that's how the real pipeline
  works; it does not accept a pre-built model object) -- Part B's model is a
  disposable inspection copy, Part C's is the official one.
- **Part D (Cell 13)**: **controlled, opt-in max-computational-power launch**
  -- auto-sizes worker/thread counts from live CPU/RAM headroom (same
  reasoning used manually throughout this project's campaign monitoring) and
  calls `run_all_scenarios_parallel()`, the exact real parallel-campaign
  function -- not a subprocess wrapper, not a reimplementation. Requires an
  explicit confirmation flag; does nothing if left at its default.

**Every import in Cell 1 is individually verified** (imported, existence
checked, source file printed) -- if anything here doesn't match what
`scenario_runner.py` actually imports, Cell 1 will tell you exactly where.

### Cell 1 — Imports, individually verified

Every name below is imported from the real module it lives in (no
reimplementation) and immediately checked: exists, is callable, and its
source file is printed so you can confirm it's the file you think it is.

In [ ]:
# =============================================================================
# Cell 1 — Setup + verified imports
# =============================================================================
import sys, os, json, time, inspect, subprocess
from pathlib import Path

_ROOT = Path(r"c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat")
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import psutil

IMPORT_CHECKS = []
def _verified_import(module_path, names):
    # "ok" means the name is genuinely present on the module -- callability is
    # NOT required, since some real exports are constants (e.g. OUT_BASE is a
    # Path, not a function). detail shows the source file for callables and
    # the repr for constants, so either way you can see exactly what you got.
    mod = __import__(module_path, fromlist=names)
    result = {}
    for name in names:
        present = hasattr(mod, name)
        obj = getattr(mod, name, None)
        if callable(obj):
            try:
                detail = inspect.getsourcefile(obj)
            except TypeError:
                detail = "?"
        else:
            detail = repr(obj)
        IMPORT_CHECKS.append({"name": name, "module": module_path, "ok": present, "detail": detail})
        result[name] = obj
    return result

_sr = _verified_import("scripts.paper_2.scenario_runner", [
    "load_scenarios_config", "_load_yaml", "_deep_merge", "_dump_yaml_tmp",
    "_apply_tes_location", "_apply_hp_location", "_apply_dsm", "_load_waste_heat",
    "_apply_spatial_temperature_offsets", "_inject_cop_series",
    "_load_outdoor_temps", "_load_hp_source_temps",
    "run_single_scenario", "run_all_scenarios_parallel", "OUT_BASE",
])
_wf = _verified_import("calion.run.workflow", ["_build_workflow_inputs", "run_workflow"])
_hk = _verified_import("calion.utils.heizkurve", ["compute_heizkurve", "check_consumer_min_temps"])
_cop = _verified_import("calion.utils.cop_wrapper", ["precompute_cop", "build_source_temperature_series"])
_sb = _verified_import("calion.models.system_builder", ["build_model"])
_ea = _verified_import("scripts.paper_2.extract_artefacts_p2", ["extract_all_p2"])

load_scenarios_config = _sr["load_scenarios_config"]; _load_yaml = _sr["_load_yaml"]
_deep_merge = _sr["_deep_merge"]; _dump_yaml_tmp = _sr["_dump_yaml_tmp"]
_apply_tes_location = _sr["_apply_tes_location"]; _apply_hp_location = _sr["_apply_hp_location"]
_apply_dsm = _sr["_apply_dsm"]; _load_waste_heat = _sr["_load_waste_heat"]
_apply_spatial_temperature_offsets = _sr["_apply_spatial_temperature_offsets"]
_inject_cop_series = _sr["_inject_cop_series"]; _load_outdoor_temps = _sr["_load_outdoor_temps"]
_load_hp_source_temps = _sr["_load_hp_source_temps"]; run_single_scenario = _sr["run_single_scenario"]
run_all_scenarios_parallel = _sr["run_all_scenarios_parallel"]; OUT_BASE = _sr["OUT_BASE"]
_build_workflow_inputs = _wf["_build_workflow_inputs"]; run_workflow = _wf["run_workflow"]
compute_heizkurve = _hk["compute_heizkurve"]; check_consumer_min_temps = _hk["check_consumer_min_temps"]
precompute_cop = _cop["precompute_cop"]; build_source_temperature_series = _cop["build_source_temperature_series"]
build_model = _sb["build_model"]
extract_all_p2 = _ea["extract_all_p2"]

_df = pd.DataFrame(IMPORT_CHECKS)
n_fail = int((~_df["ok"]).sum())
print(_df.to_string(index=False))
print(f"\n{len(_df) - n_fail}/{len(_df)} imports OK." + (f"  {n_fail} FAILED -- fix before continuing." if n_fail else ""))
assert n_fail == 0, "One or more real-pipeline functions failed to import -- do not proceed."


### Cell 2 — Pick a scenario

Mirrors `run_single_scenario()` L169-190 (id/outdir resolution, skip/feature
gates, start-of-scenario log). `SCEN_ID` is the only thing you change to pick
any scenario from either network.

In [ ]:
# =============================================================================
# Cell 2 — Pick a scenario (scenario_runner.py L150-190)
# =============================================================================
SCEN_ID = "MM-S1-HK0"   # <-- change to any id in configs/paper_2/scenarios.yaml

scen_cfg = load_scenarios_config()
scen = next(s for s in scen_cfg["scenarios"] if s["id"] == SCEN_ID)
NETWORK = scen["network"]
outdir = OUT_BASE / SCEN_ID
print(f"scen_id={SCEN_ID}  network={NETWORK}  outdir={outdir}")
print(json.dumps(scen, indent=2, ensure_ascii=False))

if (outdir / "meta.json").exists():
    print(f"\n[NOTE] {SCEN_ID} already has a completed run at {outdir}/meta.json. "
          f"The real run_single_scenario() would SKIP it here (restart safety) unless force_rerun=True. "
          f"This notebook does not auto-skip -- Cells 3-11 below are safe to inspect regardless; only "
          f"Cell 12/13 actually solve/overwrite anything, and Cell 13 passes force_rerun=True explicitly.")

assert not scen.get("requires_feature"), f"{SCEN_ID} requires an unimplemented feature: {scen.get('requires_feature')}"


### Cell 3 — Load base config + apply scenario overrides (L192-204)

In [ ]:
# =============================================================================
# Cell 3 — scenario_runner.py L192-204 (steps 1-2)
# =============================================================================
cfg_path = _ROOT / scen["config"]
if not cfg_path.exists():
    if "SB" in SCEN_ID:
        raise FileNotFoundError(f"[SKIP in real pipeline] Stadtbach config not yet available: {cfg_path}")
    raise FileNotFoundError(f"Config not found: {cfg_path}")
cfg = _load_yaml(cfg_path)
print("Loaded config:", cfg_path)

if scen.get("overrides"):
    cfg = _deep_merge(cfg, scen["overrides"])
    print("Applied scenario overrides:", json.dumps(scen["overrides"], indent=2))
else:
    print("No overrides for this scenario.")


### Cell 4 — TES/HP siting, DSM, endogenous siting (L206-240)

Same three gates as the real function, in the same order: TES location only
applies if `tes_node` is set AND it's not a baseline scenario; HP location
only if `hp_node` is set; DSM always runs (usually a no-op); endogenous
siting only if `scen["endogenous"]` is truthy.

In [ ]:
# =============================================================================
# Cell 4 — scenario_runner.py L206-240 (steps 3, 3b, 4, 4b)
# =============================================================================
if scen.get("tes_node") and not scen.get("baseline"):
    _apply_tes_location(cfg, scen, scen_cfg)
    print(f"[3] TES sited at scenario node: {scen['tes_node']}")
else:
    print("[3] TES siting: SKIPPED (baseline scenario or tes_node unset)")

if scen.get("hp_node"):
    _apply_hp_location(cfg, scen, scen_cfg)
    print(f"[3b] HP/EK sited at scenario node: {scen['hp_node']}")
else:
    print("[3b] HP/EK siting: SKIPPED (hp_node unset)")

_apply_dsm(cfg, scen, scen_cfg)
print("[4] DSM applied (no-op if this network has no dsm_consumers configured).")

if scen.get("endogenous"):
    _cands = scen_cfg.get("endogenous_candidates", {}).get(NETWORK, [])
    if not _cands:
        print(f"[4b] WARNING: endogenous scenario but no endogenous_candidates for {NETWORK}")
    else:
        cfg["endogenous_siting"] = {
            "candidates": list(_cands),
            "hp_group": list(scen_cfg.get("hp_assets", {}).get(NETWORK, [])),
            "tes_group": ["tes_main" if NETWORK == "memmingen" else "tes_sb"],
            "colocate": bool(scen.get("colocate", False)),
        }
        print(f"[4b] ENDOGENOUS SITING: candidates={_cands}, colocate={scen.get('colocate', False)}")
else:
    print("[4b] Endogenous siting: SKIPPED (not an endogenous scenario)")

print("\nAsset placements after all siting steps (nodes with a non-empty assets list):")
for nid, ncfg in cfg["network"]["nodes"].items():
    if ncfg.get("assets"):
        print(f"  {nid}: {ncfg['assets']}")


### Cell 5 — Load the data table (L246-253)

In [ ]:
# =============================================================================
# Cell 5 — scenario_runner.py L246-253 (step 5)
# =============================================================================
try:
    _first_inputs = _build_workflow_inputs([str(_dump_yaml_tmp(cfg))], overrides=None)
    table = _first_inputs.table
except Exception as exc:
    raise RuntimeError(f"[{SCEN_ID}] Failed to load data table: {exc}") from exc
print("Table rows:", len(table), "| dt_h:", _first_inputs.dt_h, "| solver:", _first_inputs.solver_name)


### Cell 6 — Heating curve T_VL(t) (L255-383)

Exact branch structure: `tvl_fix` scenarios (`BC-*`, `*-S0-TVLFIX`) get a
degenerate constant-setpoint curve; everything else looks up the real
heat-curve stage (network-keyed lookup). The T_VL_min floor
(`>= T_return + min_supply_delta_T_k`), the consumer-temperature-violation
check, and the per-asset `delta_T_scenario_k` injection into every
`geometric_storage` TES asset are all copied verbatim -- including the
worst-case (not mean) ΔT reasoning that was a real fix (2026-07-08).

In [ ]:
# =============================================================================
# Cell 6 — scenario_runner.py L255-383 (step 6)
# =============================================================================
tvl_fix = bool(scen.get("tvl_fix", False))
if tvl_fix:
    _setpoint = float(cfg.get("network", {}).get("supply_temp_c", 99.0))
    hk_stage = {"k": 0.0, "T_VL_min_c": _setpoint, "T_VL_max_c": _setpoint,
                "description": f"TVL fix: constant {_setpoint:.1f} degC (Paper-1 setpoint)"}
    print(f"[6] TVL-fix mode: constant T_VL = {_setpoint:.1f} degC")
else:
    hk_stages = scen_cfg["heat_curve_stages"]
    _first_val = next(iter(hk_stages.values())) if hk_stages else {}
    if isinstance(_first_val, dict) and NETWORK in hk_stages:
        hk_stage = hk_stages[NETWORK][scen["heat_curve_stage"]]
    else:
        hk_stage = hk_stages[scen["heat_curve_stage"]]
    print("[6] HK stage:", hk_stage)

_stage_return = hk_stage.get("T_RL_c")
if _stage_return is not None:
    cfg.setdefault("network", {})["return_temp_c"] = float(_stage_return)
    print(f"    Retrofit return temp override: T_return={_stage_return} degC")

T_aus = _load_outdoor_temps(table, cfg)
if T_aus is None and tvl_fix:
    T_aus = np.zeros(len(table))

if T_aus is not None:
    T_VL_ts = compute_heizkurve(k=hk_stage["k"], T_VL_min_c=hk_stage["T_VL_min_c"],
                                 T_VL_max_c=hk_stage["T_VL_max_c"], T_aus_ts=T_aus)
    return_temp_c = float(cfg.get("network", {}).get("return_temp_c", 60.0))
    min_delta_T = float(cfg.get("network", {}).get("min_supply_delta_T_k", 10.0))
    T_VL_min_effective = max(float(hk_stage["T_VL_min_c"]), return_temp_c + min_delta_T)
    T_VL_ts = np.maximum(T_VL_ts, T_VL_min_effective)
    if T_VL_min_effective > float(hk_stage["T_VL_min_c"]) + 0.1:
        print(f"    [FLOOR RAISED] T_VL_min: {hk_stage['T_VL_min_c']:.1f} -> {T_VL_min_effective:.1f} degC "
              f"(T_return={return_temp_c:.1f} + min_delta_T={min_delta_T:.0f}K)")
    else:
        print(f"    T_supply_min: {T_VL_min_effective:.1f} degC (T_return={return_temp_c:.1f}, "
              f"delta_T={T_VL_min_effective - return_temp_c:.1f}K)")

    _consumer_min_c = float(cfg.get("network", {}).get("consumer_min_temp_c", return_temp_c + min_delta_T))
    _violations = check_consumer_min_temps(T_VL_ts, consumer_min_temps_c={"network": _consumer_min_c})
    if _violations:
        v = _violations["network"]
        print(f"    [WARNING] CONSUMER TEMP VIOLATION: T_supply < {_consumer_min_c:.1f} degC in "
              f"{v['violations']} hours ({v['hours_below_pct']:.1f}% of year). Max deficit: {v['max_deficit_c']:.2f} K.")
    else:
        print(f"    Consumer temperature check passed: T_supply >= {_consumer_min_c:.1f} degC in all timesteps.")

    cfg.setdefault("network", {}).setdefault("heating_curve", {})
    cfg["network"]["heating_curve"]["T_supply_min_c"] = T_VL_min_effective
    cfg["network"]["heating_curve"]["T_supply_max_c"] = hk_stage["T_VL_max_c"]
    cfg.setdefault("heat_pumps", {}).setdefault("cop", {})
    cfg["heat_pumps"]["cop"]["supply_temp_min_c"] = T_VL_min_effective
    cfg["heat_pumps"]["cop"]["supply_temp_max_c"] = hk_stage["T_VL_max_c"]

    delta_T_scenario_k = round(T_VL_min_effective - return_temp_c, 2)
    for _asset_key, _asset_cfg in cfg.get("assets", {}).items():
        if _asset_cfg.get("type") == "geometric_storage":
            _asset_cfg["delta_T_scenario_k"] = delta_T_scenario_k
            print(f"    geometric_storage {_asset_key}: delta_T_scenario_k={delta_T_scenario_k:.2f} K (worst-case)")
else:
    T_VL_ts = None
    print("[6] WARNING: No outdoor temps -> heating curve not applied")

plt.figure(figsize=(10, 3))
plt.plot(T_VL_ts[:24*14] if T_VL_ts is not None else [])
plt.title(f"{SCEN_ID}: T_VL(t) — first 14 days"); plt.ylabel("degC"); plt.tight_layout(); plt.show()


### Cell 7 — COP(t): waste-heat-first, HP-source fallback (L385-447)

**This is the step the previous notebook got wrong.** The real function
tries `_load_waste_heat()` (real AVA/waste-heat data, e.g. Stadtbach) FIRST;
only if that returns `None` (e.g. Memmingen, which has no `waste_heat`
config section) does it fall back to `_load_hp_source_temps()`. Skipping the
waste-heat branch would have silently used the wrong source-temperature
series for any waste-heat-equipped network. Hot-charging (F2) is also
handled here exactly as the real function does it.

In [ ]:
# =============================================================================
# Cell 7 — scenario_runner.py L385-447 (step 7, incl. hot charging)
# =============================================================================
T_source_ts = None
waste_heat_data = _load_waste_heat(cfg, table)
if waste_heat_data is not None:
    Q_AW, T_AW, T_amb = waste_heat_data
    T_source_ts = build_source_temperature_series(Q_AW, T_AW, T_amb)
    print("[7] Source temps: REAL WASTE HEAT data (_load_waste_heat succeeded)")
else:
    T_source_ts = _load_hp_source_temps(cfg, table)
    print("[7] Source temps: FALLBACK to HP source-temp column (_load_waste_heat returned None -- "
          "this network has no waste_heat config section)")

if T_source_ts is not None and T_VL_ts is not None:
    cop_ts = precompute_cop(T_VL_ts=T_VL_ts, T_source_ts=T_source_ts, table=table, cfg=cfg, hp_type="standard")
    print(f"    Precomputed COP: mean={np.mean(cop_ts):.2f}, min={np.min(cop_ts):.2f}, max={np.max(cop_ts):.2f}")
    _inject_cop_series(cfg, cop_ts)

    if scen.get("hot_charging"):
        T_charge_c = float(hk_stage["T_VL_max_c"])
        cop_hot_ts = np.clip(
            precompute_cop(T_VL_ts=np.full(len(T_VL_ts), T_charge_c), T_source_ts=T_source_ts,
                            table=table, cfg=cfg, hp_type="standard"),
            1.01, 8.0,
        )
        _tes_key = "tes_main" if NETWORK == "memmingen" else "tes_sb"
        _hp_key = None
        for _ak, _acfg in cfg.get("assets", {}).items():
            if _acfg.get("type") == "heat_pump":
                _acfg["cop_charge_series_override"] = [round(float(c), 4) for c in cop_hot_ts]
                _hp_key = _ak
                break
        _rt = float(cfg.get("network", {}).get("return_temp_c", 60.0))
        _tes_asset = cfg.get("assets", {}).get(_tes_key, {})
        if _tes_asset.get("type") == "geometric_storage":
            _tes_asset["delta_T_scenario_k"] = round(T_charge_c - _rt, 2)
        if _hp_key is not None:
            cfg["hot_charging_coupling"] = {"hp": _hp_key, "tes": _tes_key}
            print(f"    HOT CHARGING: T_charge={T_charge_c:.1f} degC, TES dT={T_charge_c - _rt:.1f} K, "
                  f"COP_hot mean={np.mean(cop_hot_ts):.2f} (vs net {np.mean(cop_ts):.2f})")
    else:
        print("    Hot charging: not enabled for this scenario.")
else:
    print("[7] WARNING: T_source_ts or T_VL_ts is None -- COP not precomputed.")

plt.figure(figsize=(10, 3))
plt.plot(cop_ts[:24*14] if 'cop_ts' in dir() else [])
plt.title(f"{SCEN_ID}: COP(t) — first 14 days"); plt.tight_layout(); plt.show()


### Cell 8 — Spatial temperature offsets + final config assembly (L449-469)

The lightweight (McCormick-free) per-node heat-loss offset, then the exact
same final-assembly steps: inject any `extra_solver_options`, set up the
per-scenario Gurobi log file path, and disable console logging (matching the
real campaign -- see the note in Cell 12 about watching this log file
interactively).

In [ ]:
# =============================================================================
# Cell 8 — scenario_runner.py L449-469 (step 7b + final assembly)
# =============================================================================
if T_VL_ts is not None:
    _apply_spatial_temperature_offsets(cfg, T_VL_ts, SCEN_ID)
    offsets = {nid: ncfg.get("T_supply_offset_c") for nid, ncfg in cfg["network"]["nodes"].items() if "T_supply_offset_c" in ncfg}
    print("[7b] Node T_supply_offset_c:", offsets)

EXTRA_SOLVER_OPTIONS = None   # matches run_single_scenario's extra_solver_options kwarg -- set a dict to override
if EXTRA_SOLVER_OPTIONS:
    cfg.setdefault("run", {}).setdefault("solver_options", {}).update(EXTRA_SOLVER_OPTIONS)

# NOTE: real scenario_runner.py writes gurobi_{scen_id}.log (no suffix). This notebook
# appends "_notebook" so an interactive run here never clobbers a concurrently-running
# real campaign's log file for the same scenario id. Everything else is identical.
log_path = (OUT_BASE.parent / "logs" / f"gurobi_{SCEN_ID}_notebook.log").as_posix()
(OUT_BASE.parent / "logs").mkdir(parents=True, exist_ok=True)
cfg.setdefault("run", {}).setdefault("solver_options", {})["LogFile"] = log_path
cfg["run"]["solver_options"]["LogToConsole"] = 0

print("\nFinal solver_options:", json.dumps(cfg["run"]["solver_options"], indent=2))
print(f"Gurobi log will be written to: {log_path}  (tail this file to watch Cell 12's solve live)")
print("\nCfg is now IDENTICAL to what scenario_runner.py would pass to run_workflow(). Nothing solved yet.")


### Cell 9 — Build the model for inspection (NOT the official solve)

`calion.models.system_builder.build_model()` -- the same function the real
solve path uses internally. This copy is **disposable**: it exists purely so
Cells 10-11 can check the model *before* committing to a real solve. The
official run (Cell 12) rebuilds the model again inside `run_workflow()` --
that duplication is intentional and unavoidable if Cell 12 is to be an exact,
unmodified call to the real pipeline function rather than a hand-rolled
solve.

In [ ]:
# =============================================================================
# Cell 9 — build_model() for inspection only
# =============================================================================
t_build0 = time.perf_counter()
inspect_model = build_model(table, cfg, dt_h=_first_inputs.dt_h)
t_build = time.perf_counter() - t_build0
print(f"Inspection model built in {t_build:.1f} s")

n_vars = sum(1 for _ in inspect_model.component_data_objects(pyo.Var, active=True))
n_bin = sum(1 for v in inspect_model.component_data_objects(pyo.Var, active=True) if v.is_binary())
n_constr = sum(1 for _ in inspect_model.component_data_objects(pyo.Constraint, active=True))
print(f"n_vars={n_vars:,}  n_binary={n_bin:,}  n_constraints={n_constr:,}")


### Cell 10 — Full model check, part 1: every variable, constraint, and parameter group

Not a keyword-filtered subset -- **every** `Var`/`Constraint`/mutable `Param`
component on the model, grouped by component name, with instance counts.
This is what "check the whole mathematical model" means: a complete
inventory, not a sample.

In [ ]:
# =============================================================================
# Cell 10 — Exhaustive Var / Constraint / Param inventory
# =============================================================================
def _component_inventory(model, ctype):
    rows = []
    for comp in model.component_objects(ctype, active=True):
        try:
            n = sum(1 for _ in comp)
        except TypeError:
            n = 1
        extra = ""
        if ctype is pyo.Var:
            try:
                is_bin = any(v.is_binary() for v in comp.values()) if hasattr(comp, "values") else comp.is_binary()
            except Exception:
                is_bin = False
            extra = "binary" if is_bin else ""
        rows.append({"component": comp.name, "n_instances": n, "note": extra})
    return pd.DataFrame(rows).sort_values("n_instances", ascending=False)

var_inv = _component_inventory(inspect_model, pyo.Var)
constr_inv = _component_inventory(inspect_model, pyo.Constraint)
param_inv = _component_inventory(inspect_model, pyo.Param)

pd.set_option("display.max_rows", 500)
print(f"=== {SCEN_ID}: VARIABLE components ({len(var_inv)} distinct, {var_inv['n_instances'].sum():,} total instances) ===")
print(var_inv.to_string(index=False))
print(f"\n=== {SCEN_ID}: CONSTRAINT components ({len(constr_inv)} distinct, {constr_inv['n_instances'].sum():,} total instances) ===")
print(constr_inv.to_string(index=False))
print(f"\n=== {SCEN_ID}: mutable PARAMETER components ({len(param_inv)} distinct) ===")
print(param_inv.to_string(index=False))


### Cell 11 — Full model check, part 2: objective decomposition + PASS/FAIL assertions

Every named objective sub-expression from `constraint_builder.create_objective()`
(`energy_cost_expr`, `capex_cost_expr`, ..., `pressure_reg_cost_expr`) is
listed with its symbolic form. Then a set of explicit, hard assertions --
this cell **raises** if anything is structurally wrong, it does not just
print a warning.

In [ ]:
# =============================================================================
# Cell 11 — Objective decomposition + hard assertions
# =============================================================================
print(f"=== {SCEN_ID}: objective sub-expressions (symbolic, pre-solve) ===")
OBJ_TERMS = ("energy_cost", "dump_cost", "fuel_cost", "co2_cost", "demand_cost",
             "capex_cost", "activation_cost", "tie_break_cost", "storage_install_cost",
             "terminal_value", "demand_slack_cost", "return_anchor_cost", "pressure_reg_cost")
_found_terms = []
for _name in OBJ_TERMS:
    _expr = getattr(inspect_model, f"{_name}_expr", None)
    if _expr is not None:
        _found_terms.append(_name)
        _s = str(_expr.expr)
        print(f"  {_name:22s}: {_s[:150]}{' ...' if len(_s) > 150 else ''}")
print(f"\n{len(_found_terms)}/{len(OBJ_TERMS)} objective sub-expressions present.")

checks = []
def _check(name, ok, detail=""):
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})

_check("Model has an active Objective (model.obj)", hasattr(inspect_model, "obj") and inspect_model.obj.active)
_check("n_vars > 0", n_vars > 0, f"n_vars={n_vars}")
_check("n_constraints > 0", n_constr > 0, f"n_constraints={n_constr}")
_check("At least one binary variable exists (this is a MILP, not an LP)", n_bin > 0, f"n_binary={n_bin}")
_check("network_manager attached to model", getattr(inspect_model, "_network_manager", None) is not None)
_check("All objective sub-expressions present", len(_found_terms) == len(OBJ_TERMS),
       f"missing: {sorted(set(OBJ_TERMS) - set(_found_terms))}")

# Spot-check: a few config values actually made it into the built model unmangled.
nm = getattr(inspect_model, "_network_manager", None)
if nm is not None:
    _cfg_gas = cfg.get("fuels", {}).get("gas", {}).get("price_eur_mwh")
    _check("Config gas price is a real number", isinstance(_cfg_gas, (int, float)), f"fuels.gas.price_eur_mwh={_cfg_gas}")
    # _pressure_drop_enabled is a @property on NetworkManager (network_manager.py L98-101) --
    # getattr already evaluates it to a bool, it is NOT a method to call.
    _pdrop = getattr(nm, "_pressure_drop_enabled", None)
    _check("network_manager exposes pressure_drop_enabled (bool)", isinstance(_pdrop, bool),
           f"pressure_drop_enabled={_pdrop}")

checks_df = pd.DataFrame(checks)
print("\n" + checks_df.to_string(index=False))
n_fail = int((checks_df["status"] == "FAIL").sum())
print(f"\n{len(checks_df) - n_fail}/{len(checks_df)} checks passed.")
assert n_fail == 0, f"{n_fail} model check(s) FAILED -- see table above. Do not trust a solve of this model yet."
print("\nAll checks passed. Model is structurally sound -- safe to proceed to the real solve (Cell 12).")


### Cell 12 — THE REAL SOLVE (exact `run_workflow()` call, not a reconstruction)

This is `scenario_runner.py` L460-556, copied exactly: dump the final `cfg`
to a temp YAML, call `run_workflow([str(tmp_cfg_path)])` -- the same function
`_pf_step()` -> `_solve_scenario()` -> `build_model()` + real Gurobi solve
that the actual campaign uses -- then run the same incumbent/status
classification logic (a `maxTimeLimit` run with zero incumbents is `NOT`
reported as `ok`; this is what fixed the O-7 zero-cost-artefact bug), then
extract artefacts via the real `extract_all_p2()`.

**This is opt-in and can take up to 24h for a Memmingen scenario at the real
campaign's `MIPGap`/`TimeLimit`** (see the scenario's own `run.solver_options`,
already baked into `cfg` from Cell 3). Watch `log_path` from Cell 8 (e.g.
`Get-Content -Wait <path>` in PowerShell, or `tail -f` in bash) for live
Gurobi output, since `LogToConsole=0` matches the real campaign and this
cell's own output will otherwise look silent while solving.

In [ ]:
# =============================================================================
# Cell 12 — scenario_runner.py L460-556 (steps 8-9 + incumbent classification)
# =============================================================================
RUN_REAL_SOLVE = False   # <-- set True to actually launch. Defaults to False on purpose.

if not RUN_REAL_SOLVE:
    print("RUN_REAL_SOLVE is False -- not launching. Set it True and re-run this cell when ready.")
else:
    tmp_cfg_path = _dump_yaml_tmp(cfg)
    t0 = time.perf_counter()
    try:
        wf = run_workflow([str(tmp_cfg_path)])
        if os.environ.get("CALION_DEBUG_COSTS"):
            print("DEBUG_COSTS", SCEN_ID, dict(wf.pf_result.costs) if wf.pf_result else None)
        elapsed = time.perf_counter() - t0
        print(f"[{SCEN_ID}] Solved in {elapsed:.1f} s")
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        print(f"[{SCEN_ID}] Solve failed after {elapsed:.1f} s: {exc}")
        raise
    finally:
        try:
            tmp_cfg_path.unlink()
        except OSError:
            pass

    outdir.mkdir(parents=True, exist_ok=True)
    try:
        extract_all_p2(SCEN_ID, cfg, wf, elapsed, outdir, scen)
        print(f"Artefacts extracted -> {outdir}")
    except Exception as exc:
        print(f"[{SCEN_ID}] Artefact extraction failed: {exc}")

    solver_meta = {}
    obj_val = None
    pf = wf.pf_result
    if pf is not None:
        solver_meta = getattr(pf, "solver", {}) or {}
        summ = getattr(pf, "summary", {}) or {}
        obj_section = summ.get("objective", {}) if hasattr(summ, "get") else {}
        for _k in ("OBJ_value_EUR", "Model_OBJ_value_EUR"):
            if isinstance(obj_section, dict) and obj_section.get(_k) is not None:
                obj_val = float(obj_section[_k]); break

    term_cond = str(solver_meta.get("termination_condition", "")).lower()
    sol_count = solver_meta.get("solution_count", None)

    if sol_count is not None and sol_count <= 0:
        RESULT = {"id": SCEN_ID, "status": "no_incumbent", "solve_s": round(elapsed, 1),
                  "obj_eur": None, "termination": term_cond or None, "outdir": str(outdir)}
        print(f"[{SCEN_ID}] NO INCUMBENT (termination={term_cond}) -- NOT a usable result.")
    else:
        if term_cond and ("maxtimelimit" in term_cond or "aborted" in term_cond):
            print(f"[{SCEN_ID}] Hit the time limit but has a usable incumbent (termination={term_cond}, "
                  f"obj={obj_val:.0f} EUR) -- MIP gap may exceed the target.")
        RESULT = {"id": SCEN_ID, "status": "ok", "solve_s": round(elapsed, 1),
                  "obj_eur": obj_val, "termination": term_cond or None, "outdir": str(outdir)}

    print("\nRESULT:", json.dumps(RESULT, indent=2))


### Cell 13 — Controlled max-computational-power launch

Auto-sizes `--workers`/`--gurobi-threads` from **live** CPU/RAM headroom
(same reasoning applied manually throughout this project's campaign
monitoring: leave overhead for the OS, budget per-worker RAM generously
since `Cuts=2` solves reveal their real footprint slowly, target the
requested ceiling without exceeding it). Calls `run_all_scenarios_parallel()`
directly -- the exact real parallel-campaign function (`ProcessPoolExecutor`
over `run_single_scenario`), not a subprocess/CLI wrapper. Opt-in, same
pattern as Cell 12: does nothing until you explicitly enable it.

In [ ]:
# =============================================================================
# Cell 13 — Auto-sized, opt-in parallel launch (the real campaign function)
# =============================================================================
LAUNCH_MAX_POWER = False        # <-- set True to actually launch
LAUNCH_SCENARIO_IDS = [SCEN_ID] # <-- or a list of ids, e.g. ["MM-S1-HK0", "MM-S1-HK1", "MM-S1-HK2"]
TARGET_RAM_FRACTION = 0.90
TARGET_CPU_FRACTION = 0.90
RAM_PER_WORKER_GB_ESTIMATE = 10.0   # conservative -- Cuts=2 solves have been observed up to ~18-20 GB/worker
GUROBI_THREADS_PER_WORKER = 4

def compute_autoscale():
    vm = psutil.virtual_memory()
    total_gb = vm.total / (1024**3)
    avail_gb = vm.available / (1024**3)
    logical_cpus = os.cpu_count() or 4

    ram_budget_gb = total_gb * TARGET_RAM_FRACTION
    ram_headroom_gb = max(0.0, ram_budget_gb - (total_gb - avail_gb))
    workers_by_ram = max(1, int(ram_headroom_gb // RAM_PER_WORKER_GB_ESTIMATE))

    cpu_budget_threads = int(logical_cpus * TARGET_CPU_FRACTION)
    workers_by_cpu = max(1, cpu_budget_threads // GUROBI_THREADS_PER_WORKER)

    n_workers = max(1, min(workers_by_ram, workers_by_cpu, len(LAUNCH_SCENARIO_IDS)))
    return {
        "logical_cpus": logical_cpus, "total_ram_gb": round(total_gb, 1),
        "available_ram_gb": round(avail_gb, 1), "ram_headroom_gb": round(ram_headroom_gb, 1),
        "workers_by_ram": workers_by_ram, "workers_by_cpu": workers_by_cpu,
        "n_workers": n_workers, "gurobi_threads_per_worker": GUROBI_THREADS_PER_WORKER,
        "total_threads_used": n_workers * GUROBI_THREADS_PER_WORKER,
    }

scale = compute_autoscale()
print("Autoscale plan (live system state):")
print(json.dumps(scale, indent=2))
print(f"\nWould launch {len(LAUNCH_SCENARIO_IDS)} scenario(s) with "
      f"{scale['n_workers']} worker(s) x {scale['gurobi_threads_per_worker']} Gurobi threads each.")

if not LAUNCH_MAX_POWER:
    print("\nLAUNCH_MAX_POWER is False -- not launching. Set it True and re-run this cell when ready.")
else:
    print(f"\nLaunching run_all_scenarios_parallel({LAUNCH_SCENARIO_IDS}, "
          f"max_workers={scale['n_workers']}, gurobi_threads={scale['gurobi_threads_per_worker']}) ...")
    results = run_all_scenarios_parallel(
        LAUNCH_SCENARIO_IDS,
        max_workers=scale["n_workers"],
        gurobi_threads=scale["gurobi_threads_per_worker"],
        force_rerun=True,
    )
    print(json.dumps(results, indent=2))


### Interpretation

- **Parts A/B (Cells 1-11) cost nothing** -- run them any time to inspect any
  scenario's exact config, exact equations, and exact model structure before
  deciding whether to solve it.
- **Cell 12 IS the campaign** -- its result is not an approximation or a
  notebook-side reconstruction; it is produced by calling the identical
  `run_workflow()` function the CLI campaign runner calls, on the identical
  fully-prepared `cfg`.
- **Cell 13 is how to reproduce what was done manually during this project's
  campaign monitoring** (incrementally raising worker/thread counts toward a
  RAM/CPU ceiling) -- but automated and driven by the live `psutil` reading
  at the moment you run it, not a fixed guess.